In [1]:
from xgboost import XGBRegressor

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df=pd.read_csv("data.csv")
df=df.drop(["country","street","date","sqft_above","sqft_basement","yr_built","yr_renovated","waterfront",],axis=1)

from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(drop="first",
                  sparse_output=False)
encoded=ohe.fit_transform(df[["city","statezip"]])

encoded_df=pd.DataFrame(encoded,
                        columns=ohe.get_feature_names_out(["city","statezip"])
                       )

df=pd.concat(
    [df.drop(["city","statezip"],axis=1),encoded_df], axis=1)
    

q99=df["price"].quantile(0.99)
df=df[df["price"]<=q99]
df["price"]=np.log1p(df["price"])
df=df[df["price"]>0]

x=df.drop("price",axis=1)

y=df["price"]


from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42)


model=XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
    
)

model.fit(x_train,y_train)

print("r2 = " ,model.score(x_test,y_test))

r2 =  0.7724822837711207


In [6]:
y_pred=model.predict(x_test)
train_acc=model.score(x_train,y_train)
test_acc=model.score(x_test,y_test)
print("train accu=" ,train_acc)
print("test_accu = ",test_acc)



from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
print("mae = ", mean_absolute_error(y_test,y_pred))
print("mse = " ,mean_squared_error(y_test,y_pred))
print("r2 = " , r2_score(y_test,y_pred))

train accu= 0.8667014240973459
test_accu =  0.7724822837711207
mae =  0.16193335489789637
mse =  0.05511595751489363
r2 =  0.7724822837711207


In [7]:
from sklearn.model_selection import cross_val_score
scores=cross_val_score(
    model,
    x,
    y,
    cv=5,
    scoring='r2'
)
print("scores",scores)
print(scores.mean())

scores [0.81163432 0.8152485  0.83366056 0.83254597 0.67415422]
0.7934487138916821


In [13]:
import joblib

joblib.dump(model, "house_price_model.pkl")
joblib.dump(ohe, "encoder.pkl")
joblib.dump(x.columns.tolist(), "model_columns.pkl")






['model_columns.pkl']

In [10]:
import os
print("model.pkl" in os.listdir())

True


In [12]:
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,view,condition,city_Auburn,city_Beaux Arts Village,...,statezip_WA 98155,statezip_WA 98166,statezip_WA 98168,statezip_WA 98177,statezip_WA 98178,statezip_WA 98188,statezip_WA 98198,statezip_WA 98199,statezip_WA 98288,statezip_WA 98354
0,12.653962,3.0,1.50,1340,7912,1.5,0,3,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,12.742569,3.0,2.00,1930,11947,1.0,0,4,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,12.948012,3.0,2.25,2000,8030,1.0,0,4,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,13.217675,4.0,2.50,1940,10500,1.0,0,4,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,13.102163,2.0,1.00,880,6380,1.0,0,3,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
print("Bedrooms:", df["bedrooms"].min(), df["bedrooms"].max())
print("Bathrooms:", df["bathrooms"].min(), df["bathrooms"].max())
print("Sqft Living:", df["sqft_living"].min(), df["sqft_living"].max())
print("Sqft Lot:", df["sqft_lot"].min(), df["sqft_lot"].max())
print("Floors:", df["floors"].min(), df["floors"].max())
print("View:", df["view"].min(), df["view"].max())
print("Condition:", df["condition"].min(), df["condition"].max())

Bedrooms: 0.0 9.0
Bathrooms: 0.0 5.75
Sqft Living: 370 7320
Sqft Lot: 638 1074218
Floors: 1.0 3.5
View: 0 4
Condition: 1 5


In [15]:
df = pd.read_csv("data.csv")

df.columns = df.columns.str.strip()

df["city"] = df["city"].astype(str).str.strip()
df["statezip"] = df["statezip"].astype(str).str.strip()